## 1. Configuração de Ambiente

Iniciei o processo definindo as variáveis de ambiente e os caminhos para o catálogo `medalhao_credit`. Optei por garantir a criação do schema `silver` logo no início com `CREATE SCHEMA IF NOT EXISTS` para garantir a idempotência do notebook (ou seja, ele pode ser rodado múltiplas vezes sem falhar).

In [0]:
%sql

CREATE CATALOG IF NOT EXISTS medalhao_credit;

In [0]:
%sql

USE CATALOG medalhao_credit;

CREATE SCHEMA IF NOT EXISTS silver_credit;

In [0]:
%sql
USE SCHEMA silver_credit;

%md
### 1.1 Importação de Bibliotecas
Iniciei o desenvolvimento importando as funções essenciais do **PySpark**.
Selecionei especificamente as funções de transformação (`col`, `trim`, `regexp_replace`, etc.) e os tipos de dados (`StringType`, `LongType`, etc.) que seriam necessários para aplicar as regras de negócio, a limpeza de texto e a normalização do *schema* nas etapas seguintes.

In [0]:
from pyspark.sql.functions import col, trim, regexp_replace, initcap, when, lower, lit, udf
from pyspark.sql.types import StringType, IntegerType, LongType, TimestampType

## 2. Configuração e Leitura da Bronze
Defini as variáveis de ambiente `catalogo`, `bronze_db_name` e `silver_db_name` para organizar os caminhos do *Data Lake*.
Em seguida, realizei a leitura da tabela bruta (`df_bronze`). Como o arquivo original não possuía cabeçalho (gerando colunas genéricas como `_c0`), preparei o DataFrame para as transformações seguintes.

In [0]:
catalogo = "medalhao_credit"
bronze_db_name = "bronze_credit"
silver_db_name = "silver_credit"


print(f"Ambiente configurado: {catalogo}.{silver_db_name}")

In [0]:
# Leitura da tabela Bronze de Clientes
df_bronze = spark.table(f"{catalogo}.{bronze_db_name}.clientes")

display(df_bronze.limit(5))

## 2. Tratamento do ID Cliente (CPF)
Tratei a coluna `_c0`, renomeando-a para `id_cliente`.
Optei pela tipagem `long` para suportar os 11 dígitos do CPF sem truncagem. Apliquei a remoção de duplicados (`dropDuplicates`) e filtrei valores nulos, garantindo a unicidade da chave primária.

In [0]:
df_step_id = (
    df_bronze
    .withColumnRenamed("_c0", "id_cliente")
    
    # Tipagem Long (CPFs são números grandes)
    .withColumn("id_cliente", col("id_cliente").cast("long"))
    
    # Regras de Integridade
    .dropDuplicates(["id_cliente"])
    .filter(col("id_cliente").isNotNull())
)

display(df_step_id)

## 3. Tratamento do Nome
Tratei a coluna `_c1`, renomeando-a para `nome_cliente`.
Apliquei as funções `trim` para remover espaços extra e `initcap` para padronizar o texto em *Title Case* (ex: "João Silva"), corrigindo possíveis inconsistências de maiúsculas/minúsculas vindas da origem.

In [0]:
df_step_nome = (
    df_step_id # Continua do passo anterior
    .withColumnRenamed("_c1", "nome_cliente")
    
    # Padronização de Texto
    .withColumn("nome_cliente", trim(col("nome_cliente")))
    .withColumn("nome_cliente", initcap(col("nome_cliente")))
)

display(df_step_nome)

## 4. Tratamento e Validação do E-mail
Tratei a coluna `_c2`, renomeando-a para `email_cliente`.
Padronizei todos os caracteres para minúsculo (`lower`). Implementei uma regra de qualidade simples: se o e-mail não contiver o caractere "@", o valor é convertido para `null` (considerado inválido), evitando "lixo" nos dados de contacto.

In [0]:
df_step_email = (
    df_step_nome # Continua do passo anterior
    .withColumnRenamed("_c2", "email_cliente")
    
    # Padronização (Minúsculo)
    .withColumn("email_cliente", trim(lower(col("email_cliente"))))
    
    # Validação de Formato
    .withColumn("email_cliente", 
                when(col("email_cliente").contains("@"), col("email_cliente"))
                .otherwise(None))
)

display(df_step_email)

## 5. Tratamento da Região
Tratei a coluna `_c3`, renomeando-a para `regiao`.
Apliquei a limpeza de espaços e a padronização visual (*Title Case*) para garantir a consistência nos filtros e agrupamentos geográficos na camada Gold.

In [0]:
df_step_regiao = (
    df_step_email # Continua do passo anterior
    .withColumnRenamed("_c3", "regiao")
    
    # Padronização
    .withColumn("regiao", initcap(trim(col("regiao"))))
)

display(
    df_step_regiao
    .select('regiao')
    .distinct()
)

display(df_step_regiao)

## 6. Tratamento e Sanidade da Idade
Tratei a coluna `_c4`, renomeando-a para `idade`.
Forcei a conversão para o tipo Inteiro (`int`) e apliquei uma regra de sanidade (*sanity check*): valores negativos ou absurdamente altos (acima de 120 anos) foram convertidos para nulo, garantindo métricas demográficas fiáveis.

In [0]:
df_step_idade = (
    df_step_regiao # Continua do passo anterior
    .withColumnRenamed("_c4", "idade")
    
    # Tipagem
    .withColumn("idade", col("idade").cast("int"))
    
    # Regra de Sanidade (0 < Idade < 120)
    .withColumn("idade", 
                when((col("idade") > 0) & (col("idade") < 120), col("idade"))
                .otherwise(None))
)

display(df_step_idade)

## 7. Auditoria e Gravação Silver
Finalizei o processo adicionando os metadados de controle. Preservei a `data_ingestao` original (renomeada para `_bronze`) e criei a `data_ingestao_silver` com o *timestamp* atual.
Gravei a tabela processada no catálogo `medalhao_credit`, esquema `silver`, em formato Delta, garantindo a persistência e qualidade dos dados de clientes.

In [0]:
# Preparação Final
df_final = df_step_idade


# Definição do Caminho
tabela_destino = f"{catalogo}.{silver_db_name}.clientes"

# Gravação
(
    df_final.write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable(tabela_destino)
)

print(f"✅ Tabela salva com sucesso em: {tabela_destino}")
display(df_final)